# Week 20: MLOps - CI/CD, Monitoring, and LLM Observability

## Where we are

Last week you turned the fraud classifier into a production asset: a SageMaker training job, an MLflow run, a model registered in the Model Registry, a deployed endpoint, and a `classify_with_finetuned_model` tool plugged into `week19_supervisor`. That is the artifact. This week we operate it.

## Learning objectives

By the end of this lab you will be able to:

1. Add OpenTelemetry-based tracing to a Strands agent and view per-decision traces in Langfuse without modifying agent code
2. Enable data capture on a SageMaker endpoint and schedule a Model Monitor job
3. Detect statistical drift between a baseline Spark dataset and a fresh batch
4. Create a CloudWatch alarm on endpoint latency that notifies an SNS topic
5. Explain when to reach for LiteLLM versus a full agent framework

## The mental model

Week 18 measured RAG quality OFFLINE on a fixed eval set with RAGAS. That tells you what the system can do in a lab. Week 20 measures the SAME system ONLINE on live traffic with Langfuse, Model Monitor, and CloudWatch. That tells you what the system is actually doing right now, in production, on real users.

## Environment Setup

**Platform**: Azure Databricks (plain Runtime 15.4 LTS, Python 3.10 - not the ML runtime).

The next code cell uses `%pip install` (a Databricks magic) to install or upgrade the libraries this notebook needs. The default cluster `boto3` predates the Bedrock Converse API, so we upgrade it here. After the install finishes, `dbutils.library.restartPython()` restarts the Python kernel so the new versions are picked up - once it restarts, re-run from the top.

**Libraries installed**:

- `numpy<2`, `pandas<2` (pinned first to protect the runtime pyarrow)
- `boto3>=1.36` (Bedrock Converse API support)
- `sagemaker>=2.230,<3` (v3 breaks `from sagemaker import get_execution_role`)
- `strands-agents>=1.37,<2`
- `strands-agents-tools>=0.2`
- `opentelemetry-api`, `opentelemetry-sdk`, `opentelemetry-exporter-otlp` (required by Strands' OTLP exporter in Part 1; not pulled transitively)
- `litellm>=1.50` (unified LLM interface, Part 5)
- `langfuse>=2.50,<3` (observability backend, Part 1)

**Already on the cluster** (do NOT pip-install these - reinstalling them breaks the DBR REPL): `pyspark`.

Run the cell below, restart the kernel when prompted, then continue.


In [ ]:
# Install/upgrade the libraries this notebook needs. Safe to re-run.
# Databricks will ask you to restart the kernel afterwards - restartPython does that.
# This is plain DBR 15.4 LTS (not the ML runtime). numpy<2/pandas<2 are pinned
# FIRST to protect the runtime pyarrow so no transitive dependency bumps numpy to
# 2.x and crashes the kernel. The opentelemetry-exporter-otlp package is REQUIRED
# by StrandsTelemetry().setup_otlp_exporter() in Part 1; strands-agents does not
# pull it transitively.
%pip install --quiet "numpy<2" "pandas<2" "boto3>=1.36" "sagemaker>=2.230,<3" "strands-agents>=1.37,<2" "strands-agents-tools>=0.2" "opentelemetry-api" "opentelemetry-sdk" "opentelemetry-exporter-otlp" "litellm>=1.50" "langfuse>=2.50,<3"

dbutils.library.restartPython()

# Verify versions (use importlib.metadata - never pkg.__version__).
from importlib.metadata import version

for pkg in ["boto3", "sagemaker", "strands-agents", "litellm", "langfuse",
            "opentelemetry-exporter-otlp"]:
    try:
        print(f"{pkg:30s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:30s} NOT INSTALLED ({e})")


In [ ]:
# Standard MLOps setup on Databricks: pull secrets, build boto3 clients.

import os
import json
import base64
import boto3

# Per-student AWS keys live in aws-course-creds-NN; class-wide config is in
# aws-course-shared. Derive this student's scope from the Databricks identity.
_user = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().userName().get()
)
_num = _user.split("@")[0].split("-")[1] if _user.startswith("student-") else "01"
creds_scope = f"aws-course-creds-{_num}"

# Long-lived IAM-user keys (no AWS_SESSION_TOKEN).
AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope=creds_scope, key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope=creds_scope, key="aws-secret-access-key")
AWS_REGION            = dbutils.secrets.get(scope="aws-course-shared", key="aws-region")

os.environ["AWS_ACCESS_KEY_ID"] = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_REGION"] = AWS_REGION
os.environ["AWS_DEFAULT_REGION"] = AWS_REGION
# LiteLLM (Part 5) reads AWS_REGION_NAME specifically.
os.environ["AWS_REGION_NAME"] = AWS_REGION

# Clients used across the notebook
sagemaker_client = boto3.client("sagemaker", region_name=AWS_REGION)
sm_runtime = boto3.client("sagemaker-runtime", region_name=AWS_REGION)
cloudwatch = boto3.client("cloudwatch", region_name=AWS_REGION)
s3 = boto3.client("s3", region_name=AWS_REGION)

# Carried over from Week 19
ENDPOINT_NAME = "fraud-classifier-endpoint"
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("AWS region:", AWS_REGION)
print("Endpoint:", ENDPOINT_NAME)
print("Bedrock model:", BEDROCK_MODEL_ID)

In [ ]:
# Pre-flight: fail loud now if anything we depend on is missing.

# 1. Endpoint is in service
resp = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)
assert resp["EndpointStatus"] == "InService", (
    f"Endpoint {ENDPOINT_NAME} is {resp['EndpointStatus']}. "
    "Ask your instructor to redeploy the Week 19 endpoint."
)
print("Endpoint status:", resp["EndpointStatus"])

# 2. Bedrock LLM responds
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)
try:
    bedrock_runtime.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "ping"}]}],
        inferenceConfig={"maxTokens": 10, "temperature": 0},
    )
    print("Bedrock LLM probe: ok")
except Exception as e:
    print("Ask your instructor to enable Bedrock access for", BEDROCK_MODEL_ID)
    raise

print("Pre-flight checks passed.")

In [ ]:
# Week 19 fraud-investigation supervisor, rebuilt INLINE so this notebook is
# fully self-contained (no %run, no external helper notebook). It is the same
# agent we operated in Week 19: a Strands Agent backed by a Bedrock model,
# wired to a set of fraud-investigation tools. We trace THIS agent in Part 1 -
# we do NOT change it once Langfuse is wired.
#
# The supervisor has three tools:
#   1. classify_with_finetuned_model - calls the Week 19 SageMaker endpoint
#   2. check_fraud_policy            - looks up the Bread Financial fraud rules
#   3. score_transaction_risk        - a small deterministic heuristic check
# The LLM decides which tools to call and in what order; each tool call shows
# up as its own span in Langfuse.

import json
from strands import Agent, tool
from strands.models import BedrockModel


@tool
def classify_with_finetuned_model(narrative: str) -> str:
    """Classify a transaction narrative as fraud or not_fraud.

    Invokes the Week 19 fine-tuned SageMaker endpoint (a HuggingFace
    DistilBERT text classifier). Pass the free-text transaction narrative;
    the endpoint returns the predicted label and a confidence score.

    Args:
        narrative: Free-text description of the transaction.

    Returns:
        The raw JSON string returned by the endpoint, e.g.
        '[{"label": "fraud", "score": 0.93}]'.
    """
    out = sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": narrative}),
    )
    return out["Body"].read().decode()


@tool
def check_fraud_policy(topic: str) -> str:
    """Look up the Bread Financial fraud-handling policy for a topic.

    Use this to find the documented rule for a situation before recommending
    an action, so the recommendation is grounded in policy and auditable.

    Args:
        topic: A short policy topic such as 'high amount', 'cross border',
            'new customer', 'velocity', or 'card not present'.

    Returns:
        The policy text for that topic, or a default if none matches.
    """
    policy = {
        "high amount": (
            "Transactions over 1000 USD require a step-up review and a "
            "second approver before the funds are released."
        ),
        "cross border": (
            "Cross-border merchant transactions get a +1 risk weight and "
            "must be checked against the sanctioned-country list."
        ),
        "new customer": (
            "Accounts under 30 days old route to manual review regardless "
            "of the classifier score."
        ),
        "velocity": (
            "More than 3 transactions in 10 minutes from one account is a "
            "velocity flag; block pending customer contact."
        ),
        "card not present": (
            "Card-not-present purchases above 250 USD require a one-time "
            "passcode confirmation."
        ),
    }
    return policy.get(
        topic.lower(),
        "No specific policy for that topic; apply the standard risk score.",
    )


@tool
def score_transaction_risk(amount_usd: float, is_cross_border: bool,
                           account_age_days: int) -> str:
    """Compute a quick deterministic risk score for a transaction.

    This is a cheap heuristic the supervisor can use ALONGSIDE the fine-tuned
    classifier - the classifier reads the narrative text, this reads the
    structured fields. Combining a model signal with a rules signal is a
    common fraud-investigation pattern.

    Args:
        amount_usd: Transaction amount in US dollars.
        is_cross_border: True if the merchant is in a different country.
        account_age_days: Age of the customer account in days.

    Returns:
        A human-readable line with the numeric score and a LOW/MEDIUM/HIGH
        risk band.
    """
    score = 0
    if amount_usd > 1000:
        score += 2
    elif amount_usd > 250:
        score += 1
    if is_cross_border:
        score += 1
    if account_age_days < 30:
        score += 2
    band = "LOW" if score <= 1 else "MEDIUM" if score <= 3 else "HIGH"
    return f"risk_score={score} band={band}"


# The Bedrock-backed model. BedrockModel picks up the AWS credentials that
# Cell 1 exported to the environment; region_name keeps it pinned.
supervisor_model = BedrockModel(
    model_id=BEDROCK_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

# A clear, role-specific system prompt. This is what makes the agent a
# fraud-investigation SUPERVISOR rather than a generic chatbot.
SUPERVISOR_SYSTEM_PROMPT = (
    "You are a fraud investigation supervisor at Bread Financial. "
    "For each transaction you are given, run a thorough investigation:\n"
    "1. Call classify_with_finetuned_model on the transaction narrative to "
    "get the model's fraud label and confidence.\n"
    "2. Call check_fraud_policy for any relevant topic (for example "
    "'high amount', 'cross border', 'new customer', 'velocity', or "
    "'card not present').\n"
    "3. When you know the amount, whether it is cross-border, and the "
    "account age, call score_transaction_risk for a second, rules-based "
    "signal.\n"
    "Then weigh the model signal against the policy and the risk score and "
    "give a final recommendation of APPROVE, REVIEW, or BLOCK, followed by "
    "one concise sentence of justification that cites the evidence you used."
)

# Build the agent. The variable name week19_supervisor is the contract every
# downstream cell depends on - do not rename it.
week19_supervisor = Agent(
    model=supervisor_model,
    tools=[classify_with_finetuned_model, check_fraud_policy, score_transaction_risk],
    system_prompt=SUPERVISOR_SYSTEM_PROMPT,
)

print("Supervisor ready:", type(week19_supervisor).__name__)
# Agent.tool_names is the public Strands API for the registered tool names.
print("Tools:", week19_supervisor.tool_names)


## Part 1 - Langfuse observability (online traces)

### The problem

You ship `week19_supervisor`. A user complains it gave a wrong fraud decision on transaction `T-99812` at 14:32 UTC yesterday. You need to answer:

- Which tools did the supervisor call, in what order?
- What did `classify_with_finetuned_model` return?
- Which policy did `check_fraud_policy` return?
- How long did the whole decision take, and what did it cost in tokens?

Without tracing this is impossible. RAGAS does not help, because RAGAS is offline. You need a per-request flight recorder. That is Langfuse.

### How Strands + Langfuse fit together

Strands emits OpenTelemetry spans for every model call and tool call. Langfuse speaks OTLP. We point Strands at Langfuse with five lines, then re-run the agent. No code change to the agent itself.

The magic is in `StrandsTelemetry().setup_otlp_exporter()`. After that call, every `agent(prompt)` invocation produces a trace in the Langfuse UI.

In [ ]:
# Five-line wiring. After this cell every supervisor call shows up in Langfuse.

import base64
import os
from strands.telemetry import StrandsTelemetry

LANGFUSE_PUBLIC_KEY = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-public-key")
LANGFUSE_SECRET_KEY = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-secret-key")
LANGFUSE_HOST = dbutils.secrets.get(scope="aws-course-shared", key="langfuse-host")

# Langfuse OTLP endpoint expects basic auth with base64(public:secret)
LANGFUSE_AUTH = base64.b64encode(
    f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()
).decode()

os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = LANGFUSE_HOST + "/api/public/otel"
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"
# Optional: a service name groups traces in the UI
os.environ["OTEL_SERVICE_NAME"] = "bread-academy-week20"

StrandsTelemetry().setup_otlp_exporter()
print("Langfuse OTLP exporter wired. Host:", LANGFUSE_HOST)

In [ ]:
# Run a known transaction through the unchanged week19_supervisor.
# Then open Langfuse and find the trace.

sample_prompt = (
    "Investigate transaction T-10042 for customer C-7781. "
    "Use the fine-tuned classifier on the narrative, check policy, "
    "and recommend an action."
)

response = week19_supervisor(sample_prompt)
print(str(response)[:500])
print()
print("Open", LANGFUSE_HOST, "-> Traces. The trace will show:")
print("- the top-level supervisor span")
print("- each tool invocation as a child span")
print("- the bedrock-runtime converse calls with token counts")

### Lab 1 - Trace five transactions and find the slowest one (15 min)

You are the on-call data scientist. Run the supervisor on five different fraud transactions, then open Langfuse and answer three questions:

1. Which transaction took the longest end-to-end?
2. Which tool dominated the latency for that transaction?
3. What were the total input and output tokens for the slowest one?

You do NOT need to change `week19_supervisor`. You only need to invoke it five times and inspect the Langfuse UI.

Hints:
- Use a `for` loop over `test_transactions` already defined for you below.
- Strands sets the trace name from the prompt automatically. To group the five traces, give each prompt a shared tag word and filter on it in the Langfuse Traces table.
- The Langfuse UI is at `LANGFUSE_HOST`. Sort by Latency descending in the Traces table.

### Stretch

Add a custom attribute `transaction_id` to each trace using OpenTelemetry's current span. Look up `opentelemetry.trace.get_current_span().set_attribute(...)`.

### Homework extension

Pipe the production supervisor through Langfuse for 24 hours of real traffic (simulated by replaying the fraud_transactions table). Build a Databricks SQL dashboard from the Langfuse exported events to track p95 latency by tool.

In [ ]:
# Lab 1 starter. Define the five test transactions, then loop and call
# week19_supervisor. After the loop, open Langfuse to inspect traces.

test_transactions = [
    "T-10042", "T-10117", "T-10298", "T-10355", "T-10401",
]

results = None  # YOUR CODE

print("Done. Open", LANGFUSE_HOST, "-> Traces to inspect.")

In [ ]:
# SAFETY-NET for Lab 1. Run this only if you did not finish the lab.
# Skip if results is already populated.

if results is None:
    print("Using Lab 1 safety-net.")
    results = []
    for tx in test_transactions:
        prompt = (
            f"Investigate transaction {tx}. Use the fine-tuned classifier, "
            "check policy, and recommend an action."
        )
        out = week19_supervisor(prompt)
        results.append({"tx": tx, "text": str(out)[:200]})
    print(f"Ran {len(results)} transactions through the supervisor.")

### Think about it

Langfuse stores every prompt and every tool argument by default. A fraud transaction record contains a customer ID and a dollar amount. Is that PII for your jurisdiction? If it is, what would you change about this setup before pointing it at real production traffic? Consider redaction, self-hosted Langfuse, sampling, and retention windows.

## Part 2 - SageMaker data capture and Model Monitor

### What Langfuse will NOT tell you

Langfuse traces the supervisor's decisions. It does not watch the fine-tuned classifier endpoint at the input-feature level. If next month the distribution of `amount` shifts because of a new product launch, the endpoint will keep returning predictions and Langfuse will keep showing happy traces, while the classifier silently goes off the rails.

That is what SageMaker Model Monitor is for. It samples requests and responses into S3 (data capture), then a scheduled job compares the captured distribution to a baseline you computed from training data. If a feature drifts beyond a threshold, the monitoring job writes a violation report.

### Two enablement points

Data capture is configured on the EndpointConfig, not the endpoint directly. You can:
1. Set it at endpoint-creation time (cleanest, what we recommend for new endpoints)
2. Update an existing endpoint's config to enable it after the fact (what we will do today, since Week 19 created the endpoint without capture)

In [ ]:
# Enable data capture by creating a new endpoint config that points at the
# same model but adds DataCaptureConfig, then update the endpoint to use it.

import time
from datetime import datetime

bucket = dbutils.secrets.get(scope="aws-course-shared", key="course-s3-bucket")
capture_prefix = f"fraud-classifier/data-capture/{datetime.utcnow():%Y-%m-%d}"
capture_s3_uri = f"s3://{bucket}/{capture_prefix}"

# Find the model name behind the current endpoint
current_cfg_name = sagemaker_client.describe_endpoint(
    EndpointName=ENDPOINT_NAME
)["EndpointConfigName"]
current_cfg = sagemaker_client.describe_endpoint_config(
    EndpointConfigName=current_cfg_name
)
model_name = current_cfg["ProductionVariants"][0]["ModelName"]
variant_name = current_cfg["ProductionVariants"][0]["VariantName"]
instance_type = current_cfg["ProductionVariants"][0]["InstanceType"]

new_cfg_name = f"fraud-classifier-cfg-capture-{int(time.time())}"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=new_cfg_name,
    ProductionVariants=[{
        "VariantName": variant_name,
        "ModelName": model_name,
        "InitialInstanceCount": 1,
        "InstanceType": instance_type,
    }],
    DataCaptureConfig={
        "EnableCapture": True,
        "InitialSamplingPercentage": 100,
        "DestinationS3Uri": capture_s3_uri,
        "CaptureOptions": [
            {"CaptureMode": "Input"},
            {"CaptureMode": "Output"},
        ],
    },
)
sagemaker_client.update_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=new_cfg_name,
)
print("Update started. New config:", new_cfg_name)
print("Capture S3:", capture_s3_uri)

In [ ]:
# Wait for the endpoint to come back InService, then invoke a few times.
import time

while True:
    status = sagemaker_client.describe_endpoint(EndpointName=ENDPOINT_NAME)["EndpointStatus"]
    if status == "InService":
        break
    print("Endpoint status:", status, "- waiting 30s")
    time.sleep(30)

# Send 10 sample invocations. The fraud classifier is a HuggingFace
# DistilBERT text model - it expects {"inputs": "<narrative text>"}.
import json
sample_payload = json.dumps({
    "inputs": "Card-not-present purchase of 482.50 at an online electronics "
              "retailer, 2 hours after the previous transaction."
})
for i in range(10):
    sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=sample_payload,
    )

# Inspect what landed in S3. Capture can take a minute to flush.
time.sleep(60)
listing = s3.list_objects_v2(Bucket=bucket, Prefix=capture_prefix)
for obj in listing.get("Contents", [])[:5]:
    print(obj["Key"], obj["Size"], "bytes")

### Lab 2 - Baseline + scheduled monitor (15 min)

You will create a Model Monitor baseline from the training data and schedule a monitoring job on the endpoint. The lab is mostly configuration. The point is to understand the moving parts:

1. Baseline dataset (training data, in S3 as CSV)
2. `DefaultModelMonitor.suggest_baseline(...)` job (one-shot, writes constraints.json + statistics.json)
3. `monitor.create_monitoring_schedule(...)` (hourly, watches the data capture path)

You will not wait for the scheduled run to complete in class. You will start it and confirm the schedule shows up under SageMaker -> Monitoring jobs.

### Stretch

Read the generated `constraints.json` from S3 and print the inferred type and completeness for `amount`.

### Homework extension

Trigger the monitoring job by sending 1000 invocations with deliberately drifted `amount` values, then read the violation report from S3.

In [ ]:
# Lab 2 starter. Use SageMaker SDK to baseline and schedule.

from sagemaker import Session
from sagemaker.model_monitor import DefaultModelMonitor, CronExpressionGenerator
from sagemaker.model_monitor.dataset_format import DatasetFormat

sm_session = Session()
role = dbutils.secrets.get(scope="aws-course-shared", key="sagemaker-execution-role-arn")

baseline_input_s3 = f"s3://{bucket}/fraud-classifier/training/baseline.csv"
baseline_results_s3 = f"s3://{bucket}/fraud-classifier/monitoring/baseline-results"
schedule_output_s3 = f"s3://{bucket}/fraud-classifier/monitoring/schedule-results"

# Hint: build a DefaultModelMonitor (instance_type="ml.m5.xlarge"), call
# suggest_baseline(...), then create_monitoring_schedule(...).
# Note: ml.m5.xlarge has 16 GiB of memory which the baseline Processing job
# needs - smaller instances (e.g. ml.t3.medium, 4 GiB) OOM on this workload.
monitor = None  # YOUR CODE

print("Schedule created. Check SageMaker console -> Monitoring jobs.")


In [ ]:
# SAFETY-NET for Lab 2. Run only if monitor is still None.

if monitor is None:
    print("Using Lab 2 safety-net.")
    monitor = DefaultModelMonitor(
        role=role,
        instance_count=1,
        # ml.m5.xlarge (16 GiB) - smaller instances OOM on the baseline job
        # and ml.m5.xlarge has plenty of processing-job quota (verified 100).
        instance_type="ml.m5.xlarge",
        volume_size_in_gb=20,
        max_runtime_in_seconds=1800,
        sagemaker_session=sm_session,
    )
    monitor.suggest_baseline(
        baseline_dataset=baseline_input_s3,
        dataset_format=DatasetFormat.csv(header=True),
        output_s3_uri=baseline_results_s3,
        wait=True,
    )

    SCHEDULE_NAME = "fraud-classifier-hourly"

    # Idempotency: delete an existing schedule with the same name so a
    # second run of the notebook does not hit ResourceInUse.
    import time
    sm_client = sm_session.boto_session.client("sagemaker")
    try:
        sm_client.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
        print(f"Found existing schedule {SCHEDULE_NAME}, deleting before recreate.")
        sm_client.delete_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
        for _ in range(12):
            time.sleep(5)
            try:
                sm_client.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
            except sm_client.exceptions.ClientError:
                break
    except sm_client.exceptions.ClientError:
        pass

    monitor.create_monitoring_schedule(
        monitor_schedule_name=SCHEDULE_NAME,
        endpoint_input=ENDPOINT_NAME,
        output_s3_uri=schedule_output_s3,
        statistics=monitor.baseline_statistics(),
        constraints=monitor.suggested_constraints(),
        schedule_cron_expression=CronExpressionGenerator.hourly(),
        enable_cloudwatch_metrics=True,
    )
    print("Schedule created via safety-net.")


## Part 3 - Drift detection with Spark

### Two kinds of drift

- **Data drift** (covariate shift): the input feature distribution changes. Example: average `amount` jumps from $80 to $260 after a marketing campaign.
- **Concept drift**: the relationship between features and the label changes. Example: a new fraud ring uses small amounts that the old model treats as safe.

Model Monitor catches data drift on whatever the endpoint sees. But often the drift is visible upstream, in the data lake, before it ever reaches the endpoint. Spark is a natural place to detect it because the training table is already there.

In this section we compute a statistical comparison between the original training slice and a fresh weekly batch, on the features the classifier uses. If a feature's distribution moves more than a threshold, we tag it as drifted and the downstream rule is "schedule a retraining run".

In [ ]:
# Pull the original training slice and a simulated drifted batch.
# In production the "drifted" batch is last week's transactions; here
# we synthesize one so the lesson is visible in class.

from pyspark.sql import functions as F

full = spark.read.table("bread_academy.course_data.fraud_transactions")

# Use the most recent partition as the baseline slice.
latest_partition = full.agg(F.max("partition_date")).collect()[0][0]

feature_cols = ["amount", "merchant_category", "days_since_last_txn"]

baseline_df = (
    full.filter(F.col("partition_date") == latest_partition)
        .select(*feature_cols)
)

# Simulate drift: shift amounts up 3x and bias toward a new category.
drifted_df = (
    full.filter(F.col("partition_date") == latest_partition)
        .select(*feature_cols)
        .withColumn("amount", F.col("amount") * 3.0 + F.rand() * 50)
        .withColumn(
            "merchant_category",
            F.when(F.rand() > 0.5, F.lit("crypto_exchange")).otherwise(F.col("merchant_category")),
        )
)

print("Baseline partition:", latest_partition)
print("Baseline rows:", baseline_df.count())
print("Drifted rows:", drifted_df.count())
baseline_df.summary("mean", "stddev", "min", "max").show()
drifted_df.summary("mean", "stddev", "min", "max").show()

In [ ]:
# Population Stability Index (PSI) is the standard in finance for drift.
# We compute it on amount by bucketing both distributions
# into 10 deciles of the baseline and comparing fractional populations.

import math

def psi(baseline, current, column, bins=10):
    quantiles = baseline.approxQuantile(column, [i / bins for i in range(1, bins)], 0.01)
    edges = [-float("inf")] + quantiles + [float("inf")]

    def bucketize(df):
        expr = F.when(F.col(column) < edges[1], 0)
        for i in range(1, len(edges) - 1):
            expr = expr.when(F.col(column) < edges[i + 1], i)
        return df.withColumn("_b", expr).groupBy("_b").count()

    b = {r["_b"]: r["count"] for r in bucketize(baseline).collect()}
    c = {r["_b"]: r["count"] for r in bucketize(current).collect()}
    nb, nc = sum(b.values()), sum(c.values())
    score = 0.0
    for i in range(bins):
        pb = max(b.get(i, 0) / nb, 1e-6)
        pc = max(c.get(i, 0) / nc, 1e-6)
        score += (pc - pb) * math.log(pc / pb)
    return score

amount_psi = psi(baseline_df, drifted_df, "amount")
days_psi   = psi(baseline_df, drifted_df, "days_since_last_txn")
print(f"PSI amount: {amount_psi:.3f}")
print(f"PSI days_since_last_txn: {days_psi:.3f}")
print("Rule of thumb: PSI < 0.1 stable, 0.1-0.25 moderate drift, > 0.25 significant drift.")

### Lab 3 - Detect categorical drift and decide whether to retrain (15 min)

PSI works for numeric features. For categorical features like `merchant_category` we compare category frequency vectors. Your task:

1. Build a function `category_drift(baseline_df, current_df, column)` that returns the L1 distance between normalized category frequency vectors.
2. Run it on `merchant_category`.
3. Define a simple decision rule: if any numeric feature has PSI > 0.25 OR any categorical feature has L1 distance > 0.3, print "RETRAIN". Otherwise print "OK".

Hints:
- Use `groupBy(column).count()` on each DataFrame, then `toPandas()` and align on the union of category names.
- Normalize each count vector to sum to 1.
- L1 distance = sum of absolute differences.

### Stretch

Wrap the whole drift check in a function and log the drift scores as parameters to a new MLflow run tagged `drift_check`. That gives you a historical record of drift over time.

### Homework extension

Schedule this drift check as a daily Databricks job. On RETRAIN, trigger the Week 19 SageMaker training pipeline via the SageMaker SDK so the loop closes automatically.

In [ ]:
# Lab 3 starter.

def category_drift(baseline, current, column):
    # YOUR CODE
    return None  # YOUR CODE

cat_drift = None  # YOUR CODE

decision = None  # YOUR CODE

print("Categorical drift:", cat_drift)
print("Decision:", decision)

In [ ]:
# SAFETY-NET for Lab 3.

if decision is None:
    print("Using Lab 3 safety-net.")
    def category_drift(baseline, current, column):
        b = {r[column]: r["count"] for r in baseline.groupBy(column).count().collect()}
        c = {r[column]: r["count"] for r in current.groupBy(column).count().collect()}
        keys = set(b) | set(c)
        nb, nc = sum(b.values()), sum(c.values())
        return sum(abs(b.get(k, 0) / nb - c.get(k, 0) / nc) for k in keys)

    cat_drift = category_drift(baseline_df, drifted_df, "merchant_category")
    decision = "RETRAIN" if (amount_psi > 0.25 or days_psi > 0.25 or cat_drift > 0.3) else "OK"
    print("Categorical drift:", cat_drift)
    print("Decision:", decision)

## Part 4 - CloudWatch alarm on endpoint latency

Model Monitor fires on data quality. Langfuse fires on agent behavior. Neither pages you at 3am when the endpoint just becomes slow.

SageMaker endpoints emit metrics to CloudWatch in the `aws/sagemaker/Endpoints` namespace: `ModelLatency`, `Invocations`, `Invocation4XXErrors`, `Invocation5XXErrors`. We will create one alarm: if `ModelLatency` p95 exceeds 1000 ms for 5 minutes, send a message to the SNS topic the instructor pre-created. SNS fans out to email, Slack, PagerDuty, whatever.

Watch the unit: `ModelLatency` is reported in microseconds, so a 1000 ms threshold is `1000000` in the alarm config.

This part is a demo only. You watch the cell run and the alarm appear in the CloudWatch console.

In [ ]:
# Create a CloudWatch alarm wired to an SNS topic the instructor provisioned.

sns_topic_arn = dbutils.secrets.get(scope="aws-course-shared", key="sns-alerts-topic-arn")

cloudwatch.put_metric_alarm(
    AlarmName="fraud-classifier-high-latency",
    AlarmDescription="Fires when p95 ModelLatency exceeds 1000 ms for 5 minutes",
    MetricName="ModelLatency",
    Namespace="AWS/SageMaker",
    # ModelLatency is reported in MICROSECONDS. 1000 ms = 1000000 us.
    ExtendedStatistic="p95",
    Dimensions=[
        {"Name": "EndpointName", "Value": ENDPOINT_NAME},
        {"Name": "VariantName", "Value": "AllTraffic"},
    ],
    Period=60,
    EvaluationPeriods=5,
    Threshold=1000000.0,
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    AlarmActions=[sns_topic_arn],
)

print("Alarm created. Open CloudWatch -> Alarms to see it.")
print("Subscribe yourself to", sns_topic_arn, "to receive the page.")

## Part 5 - LiteLLM as a unified interface (brief)

Strands gives you agents. Langfuse gives you traces. But sometimes you do not need an agent; you just need to call an LLM and log the call. LiteLLM is the "boring" version of this pattern: a single Python function `litellm.completion(...)` that speaks to any provider, with built-in callbacks for Langfuse and others.

You will not build with LiteLLM today. You will see it work in one cell. The point is the interface: same call, any provider, free Langfuse logging.

In [ ]:
# Single demo of LiteLLM auto-logging to Langfuse. No agent involved.

import litellm

# Tell LiteLLM to log everything to Langfuse. The Langfuse env vars
# from Part 1 are still set, so it picks them up automatically.
litellm.success_callback = ["langfuse"]
litellm.failure_callback = ["langfuse"]

resp = litellm.completion(
    model="bedrock/converse/us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    messages=[
        {"role": "user", "content": "In one sentence, why is online monitoring required for ML systems?"}
    ],
    max_tokens=80,
    temperature=0,
)
print(resp.choices[0].message.content)
print()
print("This call appears in Langfuse as a Generation (not a Trace), tagged with the model name.")

### Think about it

You now have four production signals on the same fraud system:

1. Langfuse traces (per-decision, qualitative)
2. SageMaker Model Monitor (feature distribution, statistical)
3. Spark drift jobs (upstream, decision-level)
4. CloudWatch alarms (operational, time-series)

Which of these would have caught the following, in order of speed?
- The endpoint instance ran out of memory at 3:04 am
- A new fraud pattern using sub-dollar amounts started yesterday
- The supervisor started calling `policy_retriever_v2` 14 times per decision because of a prompt regression you shipped Friday
- The training-serving skew on `merchant_category` has been growing for three weeks

There is no single right ordering. The point is that each signal has a job, and none of them substitutes for the others.

## A note on CI/CD

We did not build a GitHub Actions workflow in this notebook. In a Databricks-centric MLOps shop you usually combine two pieces:

1. A GitHub Actions workflow that, on push to `main`, runs unit tests, packages the training code, and submits a Databricks Job that runs the Week 19 SageMaker training script.
2. A Databricks Workflow that, on schedule, runs the drift check from Part 3 and triggers (1) if RETRAIN is decided.

The "CI" half is testing your training code before it can run. The "CD" half is the drift check that decides when to run it. You already have both ingredients; the wiring is environment-specific homework.

## Wrap-up

You took a deployed Strands-based fraud system and put four production guardrails on it:

- Langfuse + OTEL traces on `week19_supervisor` with five lines of setup
- SageMaker data capture and a Model Monitor schedule on `fraud-classifier-endpoint`
- A PSI- and L1-based drift detector running in Spark, with a clear RETRAIN rule
- A CloudWatch alarm on endpoint latency wired to SNS
- A glimpse of LiteLLM as the simpler unified-interface pattern

## Homework

1. Finish the homework extension on Lab 1 (24h replay + p95 dashboard).
2. Finish the homework extension on Lab 3 (daily Databricks job that triggers Week 19's training pipeline on RETRAIN).
3. Read the SageMaker Model Monitor docs for the four built-in monitor types: Data Quality, Model Quality, Bias Drift, Feature Attribution Drift. Pick one of the latter three and write a 200-word note on when you would add it to this pipeline.

Next week (Week 21) we move to orchestration with Airflow / MWAA. The drift check from Lab 3 will become a DAG.